In [ ]:

# ============================================================
# STEERING INFERENCE GRID ALPHA — POLYNOMIAL
# ============================================================

import re
import random
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


# ============================================================
# PATHS
# ============================================================
# ROOT_DIR is a root directory of the project. Put the path instead of "...".
ROOT_DIR = Path(r"...")

MODEL_DIR = ROOT_DIR / "models" / "MODEL" # choose the model from the models directory
TEST_PATH = ROOT_DIR / "data" / "datasets" / "polynomial" / "polynomial_test.xlsx"
STEERING_PATH = ROOT_DIR / "steering" / "polynomial" / "steering_polynomial_Reasoner_Zero.pt" #choose polynomial steering vector file for the chosen model

OUT_PATH = ROOT_DIR / "RESULT.xlsx" #you can change the name and path of the output file or keep the default one


# ============================================================
# CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_DTYPE = torch.float16
GEN_MAX_NEW_TOKENS = 512

N_RUNS = 1

# ===== ALPHA GRID =====
# ALPHA_GRID = [1.0]
ALPHA_GRID = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5]


# ============================================================
# BLEU + NORMALIZATION (polynomial-friendly)
# ============================================================

def normalize_answer(s: str) -> str:
    if not s:
        return ""

    # remove boxed
    s = re.sub(r"\\boxed\{(.+?)\}", r"\1", s)

    # remove y= part
    s = re.sub(r"^y\s*=\s*", "", s)

    s = s.replace("\\left", "").replace("\\right", "")

    s = re.sub(r"\s+", "", s)
    return s


def tokenize_math(expr: str):
    return re.findall(r"[A-Za-z]+|\d+|\^|\+|\-|\*|\/|\(|\)|C", expr)


def compute_bleu(true: str, pred: str) -> float:
    if not true or not pred:
        return 0.0
    return sentence_bleu(
        [tokenize_math(true)],
        tokenize_math(pred),
        weights=(0.5, 0.5),
        smoothing_function=SmoothingFunction().method1
    )


# ============================================================
# UTILS
# ============================================================

def extract_boxed(text: str) -> str:
    matches = [m.start() for m in re.finditer(r"\\boxed\{", text)]
    if not matches:
        return ""
    start = matches[-1] + len(r"\boxed{")
    depth, i = 1, start
    while i < len(text) and depth:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1
    return text[start:i-1].strip()


# ============================================================
# PROMPT — POLYNOMIAL VERSION
# ============================================================

def make_prompt(eq: str) -> str:
    return (
        "You are a symbolic mathematics model.\n\n"

        "Task: compute y(x) from the given derivative y'(x).\n\n"

        "Requirements:\n"
        "- Output ONLY the final explicit polynomial y(x).\n"
        "- Do NOT output integrals.\n"
        "- Do NOT output the symbol \\int.\n"
        "- Use LaTeX.\n"
        "- Return exactly one boxed expression of the form \\boxed{y=...+C}.\n"
        "- Include +C.\n"
        "- No reasoning.\n"
        "- No explanations.\n\n"

        f"PROBLEM:\n{eq}\n\n"
        "ANSWER:\n"
    )


# ============================================================
# MODEL
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=TORCH_DTYPE,
    device_map={"": 0} if DEVICE.type == "cuda" else None,
)

model.eval()
torch.set_grad_enabled(False)


# ============================================================
# STEERING
# ============================================================

class Steering(nn.Module):
    def __init__(self, model, alpha):
        super().__init__()
        self.layers = model.model.layers
        h = model.config.hidden_size
        self.vectors = nn.Parameter(torch.zeros(len(self.layers), h, device=DEVICE))
        self.alpha = alpha
        self.handles = []

    def install(self):
        self.remove()

        def make_hook(i):
            def hook(_, __, out):
                if isinstance(out, tuple):
                    out = out[0]
                return out + (self.alpha * self.vectors[i]).to(out.dtype)
            return hook

        for i, layer in enumerate(self.layers):
            self.handles.append(layer.mlp.down_proj.register_forward_hook(make_hook(i)))

    def remove(self):
        for h in self.handles:
            try:
                h.remove()
            except:
                pass
        self.handles = []


steering = Steering(model, 1.0)
steering.install()

ckpt = torch.load(STEERING_PATH, map_location=DEVICE)
steering.vectors.data = ckpt["vectors"].to(DEVICE)

loaded_alpha = ckpt.get("alpha", 1.0)
steering.alpha = loaded_alpha

print(f"Loaded steering: {steering.vectors.shape} | trained alpha={loaded_alpha}")


# ============================================================
# GENERATE
# ============================================================

@torch.no_grad()
def generate_answer(eq, alpha):
    steering.alpha = alpha

    prompt = make_prompt(eq)
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    seq = model.generate(
        **enc,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        do_sample=False,
        use_cache=False,
    )

    txt = tokenizer.decode(seq[0], skip_special_tokens=True)
    boxed = extract_boxed(txt)

    return boxed


# ============================================================
# MAIN LOOP
# ============================================================

df = pd.read_excel(TEST_PATH)
df = df.dropna(subset=["true_answer", "equation"]).reset_index(drop=True)

all_rows = []

print(f"Dataset size: {len(df)}")

for alpha in ALPHA_GRID:

    print("\n" + "#"*120)
    print(f"ALPHA = {alpha}")
    print("#"*120)

    for i, row in tqdm(df.iterrows(), total=len(df)):

        eq = str(row["equation"])
        true_ans = str(row["true_answer"])

        bleu_off_runs = []
        bleu_on_runs = []

        for run in range(N_RUNS):

            pred_off = generate_answer(eq, alpha=0.0)
            pred_on  = generate_answer(eq, alpha=alpha)

            bleu_off = compute_bleu(normalize_answer(true_ans), normalize_answer(pred_off))
            bleu_on  = compute_bleu(normalize_answer(true_ans), normalize_answer(pred_on))

            bleu_off_runs.append(bleu_off)
            bleu_on_runs.append(bleu_on)

            print("\n" + "="*80)
            print(f"[{i}] alpha={alpha} run={run+1}")
            print("EQ:", eq)
            print("TRUE:", true_ans)
            print(f"OFF: {pred_off} | BLEU={bleu_off:.4f}")
            print(f"ON : {pred_on} | BLEU={bleu_on:.4f}")

            all_rows.append({
                "alpha": alpha,
                "eq_id": i,
                "run": run,
                "equation": eq,
                "true_answer": true_ans,
                "pred_off": pred_off,
                "pred_on": pred_on,
                "bleu_off": bleu_off,
                "bleu_on": bleu_on,
                "delta_bleu": bleu_on - bleu_off
            })

        print(f"\nAVG OFF BLEU: {np.mean(bleu_off_runs):.4f}")
        print(f"AVG ON  BLEU: {np.mean(bleu_on_runs):.4f}")


# ============================================================
# SAVE EXCEL
# ============================================================

out_df = pd.DataFrame(all_rows)
out_df.to_excel(OUT_PATH, index=False)

print("\nSaved full results to:", OUT_PATH)
